# Essai : le jeu de base avec les correctifs des codifications d'avril à août

*Journal : [`tableau_resultats.md`](tableau_resultats.md). Copie de [`essai_jeu_de_base.ipynb`](../comparatif_global/essai_jeu_de_base.ipynb), mêmes préréglages.*

**Question de l'essai** : les correctifs des codifications, appliqués au jeu de base, améliorent-ils la précision des modèles **à rappel égal** ?

**Un seul changement par rapport à `essai_jeu_de_base`** : avant tout apprentissage, les codifications d'avril à août (`PAY_6` à `PAY_2`) sont corrigées, dans cet ordre :
1. **correctifs des codifications 1** (notebook `10`, codification anticipée) : solde du mois suivant nul ou créditeur (codification saine antérieure), encours nul, impayés, décisions de l'utilisateur ;
2. **règle fixe des retards** (passage 18 du journal, abandonnée depuis dans le projet) : une facture du mois d'au moins 100 NT$ sans aucun paiement le mois suivant vaut un retard (2, ou niveau du mois précédent + 1).

`PAY_1` n'est pas touchée (le paiement d'octobre est inconnu). Aucune colonne n'est ajoutée ni retirée ; `dpnm` ne sert à aucune correction.

**Mêmes préréglages qu'`essai_jeu_de_base`** : les 30 000 clients du niveau 0 (`data/creditcard_pret_ingestion.csv`), mêmes encodages, mêmes 4 modèles, même boucle (précision moyenne, surapprentissage à 7 %), mêmes grilles de départ, même seuil optimisé (au plus 1 bon client signalé pour 1 défaut, à la marge), mêmes 5 plis (`random_state=42`). Chaque client est donc jugé sur le même pli dans les deux essais : la comparaison se fait pli par pli.

**Écart avec `essai_jeu_de_base`** :
- la variante **« Sans les `PAY_n` »** n'est pas relancée : les correctifs ne touchent que les `PAY_n`, ses données sont donc identiques à celles d'`essai_jeu_de_base` ;
- les sections « Yeh et Lien » et « comparaison avec les scénarios » ne sont pas reprises ; elles sont remplacées par la **comparaison avec `essai_jeu_de_base`** (section 6) ;
- les probabilités hors pli sont enregistrées dans le dossier local `sorties/`, pas dans `data/ML/comparatif_global/`.

⚠️ **Le test du ML est inclus**, comme dans `essai_jeu_de_base` : aucune conclusion de cet essai ne doit servir à choisir le modèle final ou ses variables.

**Étapes** :
1. chargement du jeu de base ; 1 bis. correctifs des codifications d'avril à août ;
2. pipeline, critère et boucle (repris d'`essai_jeu_de_base`) ;
3. boucle d'optimisation, variante par variante ;
4. seuil de décision optimisé et comparaison des variantes, pli par pli ;
5. ce qui porte les prédictions : importance par permutation ;
6. comparaison avec `essai_jeu_de_base` : précision à rappel égal, pli par pli.

## 1. Chargement du jeu de base et variables des trois variantes

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

original = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")   # niveau 0 du nettoyage (02_01_nettoyage.ipynb)
assert original["ID"].is_unique and not original.isna().any().any()
y_train = original["dpnm"].reset_index(drop=True)
print(f"Clients : {len(original)} ; colonnes : {original.shape[1]} (ID, 23 variables, dpnm)")
print(f"Défauts : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

Clients : 30000 ; colonnes : 25 (ID, 23 variables, dpnm)
Défauts : 6636 (22.12 %)


## 1 bis. Correctifs des codifications d'avril à août (`10` puis règle fixe des retards)

Les étapes reprennent celles du notebook `10`, puis la règle fixe des retards (identiques au `11a`). Après les correctifs du `10`, il ne doit plus rester aucune codification 1 d'avril à août ; la règle fixe des retards s'applique ensuite. Le premier tableau compte les codifications modifiées par chaque étape ; le second, par mois et par passage (codification d'origine → codification corrigée, en familles).

In [2]:
SEUIL = 100   # NT$ : règle fixe des retards (09)
MOIS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai", 6: "avril"}
PAY = [f"PAY_{n}" for n in range(1, 7)]
avant_corrections = original[PAY].copy()
journal = {}

def saine_anterieure(ligne, n):
    for k in range(n + 1, 7):
        if ligne[f"PAY_{k}"] <= 0:
            return ligne[f"PAY_{k}"]
    return None

# 10, étape 0 : PAY_n = 1 et solde du mois suivant nul ou créditeur -> codification saine antérieure
nb = 0
for n in range(6, 1, -1):
    for k in original.index[(original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n - 1}"] <= 0)]:
        v = saine_anterieure(original.loc[k], n)
        if v is not None:
            original.at[k, f"PAY_{n}"] = v; nb += 1
journal["10 : solde du mois suivant nul (codification saine antérieure)"] = nb

# 10, étape 1 : encours nul (pas de facture du mois, mois précédent sain) -> codification du mois précédent
nb = 0
for n in range(5, 1, -1):
    regle = (original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n}"] <= 0) & (original[f"PAY_{n + 1}"] <= 0)
    original.loc[regle, f"PAY_{n}"] = original.loc[regle, f"PAY_{n + 1}"]; nb += int(regle.sum())
journal["10 : encours nul"] = nb

# 10, étape 2 : impayés sur une codification 1 (facture >= 100 NT$, rien payé le mois suivant) -> retard
nb = 0
for n in range(6, 1, -1):
    regle = (original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n}"] >= SEUIL) & (original[f"PAY_AMT{n - 1}"] == 0)
    avant = original[f"PAY_{n + 1}"] if n < 6 else pd.Series(np.nan, index=original.index)
    original.loc[regle, f"PAY_{n}"] = pd.Series(np.where(avant >= 2, avant + 1, 2), index=original.index)[regle]; nb += int(regle.sum())
journal["10 : impayés sur une codification 1"] = nb

# 10, étape 3 : décisions de l'utilisateur (codification anticipée)
DECISIONS = {6783: {4: 0, 3: 0, 2: 0}, 11498: {4: 2, 3: 2, 2: 2}, 892: {3: 0, 2: 0}, 1967: {3: 0, 2: 0},
             5645: {2: 0}, 19195: {2: 2}, 25176: {2: 2}}
nb = 0
for id_client, valeurs in DECISIONS.items():
    ligne = original.index[original["ID"] == id_client]
    assert len(ligne) == 1, id_client
    for n, v in valeurs.items():
        original.loc[ligne, f"PAY_{n}"] = v; nb += 1
journal["10 : décisions de l'utilisateur"] = nb
assert int((original[PAY[1:]] == 1).sum().sum()) == 0, "il reste des codifications 1 d'avril à août"

# 09 : règle fixe des retards, d'avril à août
nb = 0
for n in range(6, 1, -1):
    regle = (original[f"PAY_{n}"] <= 1) & (original[f"BILL_AMT{n}"] >= SEUIL) & (original[f"PAY_AMT{n - 1}"] == 0)
    avant = original[f"PAY_{n + 1}"] if n < 6 else pd.Series(np.nan, index=original.index)
    original.loc[regle, f"PAY_{n}"] = pd.Series(np.where(avant >= 2, avant + 1, 2), index=original.index)[regle]; nb += int(regle.sum())
journal["09 : règle fixe des retards"] = nb
assert (original["PAY_1"] == avant_corrections["PAY_1"]).all(), "PAY_1 ne doit pas être corrigé"
assert not original.isna().any().any()

display(pd.DataFrame({"Codifications modifiées": journal}))

def famille(v):
    return np.where(v >= 2, "retard (2 et plus)", np.where(v == 1, "1", "sain (-2, -1, 0)"))

lignes = []
for n in range(6, 1, -1):
    change = original[f"PAY_{n}"] != avant_corrections[f"PAY_{n}"]
    passages = pd.Series(famille(avant_corrections.loc[change, f"PAY_{n}"])) + " → " + pd.Series(famille(original.loc[change, f"PAY_{n}"]))
    ligne = {"Mois": MOIS[n], "Codifications modifiées": int(change.sum())}
    ligne.update(passages.value_counts().to_dict())
    lignes.append(ligne)
display(pd.DataFrame(lignes).set_index("Mois").fillna(0).astype(int))
print(f"Clients avec au moins une codification modifiée : {int((original[PAY] != avant_corrections).any(axis=1).sum())} sur {len(original)}")

,Codifications modifiées
10 : solde du mois suivant nul (codification saine antérieure),0
10 : encours nul,21
10 : impayés sur une codification 1,0
10 : décisions de l'utilisateur,13
09 : règle fixe des retards,2008


,Codifications modifiées,"sain (-2, -1, 0) → retard (2 et plus)","1 → sain (-2, -1, 0)",1 → retard (2 et plus)
Mois,,,,
avril,558,558,0,0
mai,897,897,0,0
juin,520,518,1,1
juillet,23,19,3,1
août,44,16,25,3


Clients avec au moins une codification modifiée : 1698 sur 30000


## 1 ter. Variables des deux variantes

In [3]:
# ==============================================================================
# VARIABLES : les 23 variables d'origine, rangées selon leur encodage (celui des scénarios, méthode v8)
# ==============================================================================
PAY = [f"PAY_{n}" for n in range(1, 7)]

# PAY_n : nombres ordonnés (-2 à 8), mis à l'échelle (v6)
# PAY_AMTn : paiements très étirés, logarithme (v7)
# BILL_AMTn : factures, logarithme signé, car elles peuvent être négatives (v8)
# SEX, EDUCATION, MARRIAGE : catégories sans ordre, une colonne par catégorie (regroupées au niveau 0)
TYPES_ORIGINE = {
    "num": ["LIMIT_BAL", "AGE"] + PAY,
    "log": [f"PAY_AMT{n}" for n in range(1, 7)],
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "nom": ["SEX", "EDUCATION", "MARRIAGE"],
}

# Les trois variantes : colonnes gardées, chacune avec son encodage d'origine
# Variante « Sans les PAY_n » non relancée : les correctifs ne touchent que les PAY_n, ses données sont celles d'essai_jeu_de_base
VARIANTES = {
    "Complet": lambda c: True,
    "PAY_n seuls": lambda c: c in PAY,
}
TYPES_VARIANTE = {v: {t: [c for c in cols if garder(c)] for t, cols in TYPES_ORIGINE.items()} for v, garder in VARIANTES.items()}
COLONNES_VARIANTE = {v: [c for cols in types.values() for c in cols] for v, types in TYPES_VARIANTE.items()}

# Contrôles : colonnes présentes, ni identifiant ni cible, les 23 variables couvertes par la variante complète
assert sorted(COLONNES_VARIANTE["Complet"]) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))
assert (original[TYPES_ORIGINE["log"]] >= 0).all().all(), "Le logarithme ne s'applique qu'à des montants positifs ou nuls"

display(pd.DataFrame({
    "Variante": list(VARIANTES),
    "Nombre de variables": [len(COLONNES_VARIANTE[v]) for v in VARIANTES],
    "Variables": [", ".join(COLONNES_VARIANTE[v]) for v in VARIANTES],
}).set_index("Variante"))

,Nombre de variables,Variables
Variante,,
Complet,23,"LIMIT_BAL, AGE, PAY_1, PAY_2, PAY_3, PAY_4, PA..."
PAY_n seuls,6,"PAY_1, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6"


## 2. Pipeline, critère et boucle : repris des scénarios

Le pipeline et la boucle sont ceux des scénarios (`ml_13`, méthode v8), à quatre différences près : le **critère de la boucle**, la précision moyenne (`CRITERE`, au lieu de la précision au rappel minimal) ; le **seuil de surapprentissage à 7 %**, mesuré sur ce critère ; les **4 modèles** suivis ; un pipeline construit **pour chaque variante** (seules ses colonnes y entrent). Le plancher contre le sous-apprentissage ne change pas de sens : la précision moyenne d'un tri au hasard vaut le taux de défaut. Aucune valeur n'est remplacée : le jeu de base n'a pas de vide.

In [4]:
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)   # textes affichés en entier

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def preprocesseur(variante):
    # Une branche par type de colonne, seulement pour les types présents dans la variante
    return ColumnTransformer(transformers=[(nom, TRANSFORMATIONS[nom](), colonnes)
                                           for nom, colonnes in TYPES_VARIANTE[variante].items() if colonnes])

# Critère optimisé par la boucle : la précision moyenne (aire sous la courbe précision-rappel), sur tous les rappels
CRITERE = "average_precision"
scorers = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
}

# Rappel minimal des scénarios (60 %) : sert seulement à la comparaison avec les scénarios (section 6)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles de l'essai : les meilleurs des scénarios, avec leurs réglages de la première itération
MODELES_A_TESTER = ["LogisticRegression", "MLPClassifier", "RandomForest", "CatBoost"]

def modeles_de_base(variante):
    prep = preprocesseur(variante)
    tous = {
        "LogisticRegression": make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "MLPClassifier": make_pipeline(prep, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "RandomForest": make_pipeline(prep, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(prep, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

# ------------------------------------------------------------------------------
# Alertes et tableaux de scores (ceux des scénarios)
SEUIL_SURAPPRENTISSAGE = 0.07   # écart relatif train - val du critère : 7 % pour cet essai (5 % dans les scénarios) ; le double : surapprentissage fort
SEUIL_INSTABILITE = 0.02        # écart-type du critère sur les plis au-delà duquel on alerte (alerte seule, pas une contrainte)
METRIQUES = {"average_precision": "Précision moyenne", "roc_auc": "ROC AUC"}
LIBELLE_SCORE = "Précision des défauts prédits"
TAUX_DEFAUT = y_train.mean()    # précision, et précision moyenne, d'un tri au hasard

def apprend(score_val, std_val):
    # Plancher contre le sous-apprentissage : le critère doit dépasser le hasard (taux de défaut) de plus d'un écart-type
    return score_val - TAUX_DEFAUT > std_val

def alerte(ecart):
    if ecart > 2 * SEUIL_SURAPPRENTISSAGE:
        return "⚠️⚠️ SURAPPRENTISSAGE FORT"
    if ecart > SEUIL_SURAPPRENTISSAGE:
        return "⚠️ SURAPPRENTISSAGE"
    return "✅ OK"

def ligne_scores(nom, moy_val, std_val, moy_train):
    # Une ligne de tableau : chaque métrique en entraînement et en validation, le critère, son écart-type et l'alerte
    ligne = {"Modèle": nom}
    for cle, libelle in METRIQUES.items():
        ligne[f"{libelle} (train)"] = moy_train[cle]
        ligne[f"{libelle} (val)"] = moy_val[cle]
    ligne["Critère (val)"], ligne["Critère (train)"] = moy_val[CRITERE], moy_train[CRITERE]
    ligne["Écart-type critère (val)"] = std_val
    ligne["Écart train - val"] = moy_train[CRITERE] - moy_val[CRITERE]
    ligne["Écart relatif"] = ligne["Écart train - val"] / moy_val[CRITERE]
    ligne["Apprend"] = apprend(moy_val[CRITERE], std_val)
    ligne["Alerte"] = alerte(ligne["Écart relatif"]) if ligne["Apprend"] else "⚠️ N'APPREND RIEN (niveau du hasard)"
    if std_val > SEUIL_INSTABILITE:
        ligne["Alerte"] += " ; ⚠️ INSTABLE"
    return ligne

def pct(serie):
    return (serie * 100).round(1)

def parametres(params):
    # Paramètres sur une ligne : « depth 4, 5, 6 ; l2_leaf_reg 15 »
    return " ; ".join(f"{p} {', '.join(map(str, v)) if isinstance(v, list) else v}" for p, v in params.items())

def montrer(df, titre):
    display(Markdown(f"**{titre}**"))
    display(df.set_index(df.columns[0]))

In [5]:
from itertools import product
from sklearn.model_selection import GridSearchCV

TOURS_SECURITE = 15   # garde-fou contre une boucle sans fin : la boucle s'arrête d'elle-même avant ; s'il est atteint, revoir les grilles ou les bornes

# Grilles du premier tour : celles des scénarios
grilles = {
    "LogisticRegression": {"logisticregression__C": [0.01, 0.1, 1.0, 10.0]},
    "MLPClassifier": {"mlpclassifier__hidden_layer_sizes": [(32, 16), (64, 32), (32,)], "mlpclassifier__alpha": [0.01, 0.1, 1.0]},
    "RandomForest": {"randomforestclassifier__max_depth": [6, 8, 10], "randomforestclassifier__min_samples_leaf": [20, 50, 100], "randomforestclassifier__n_estimators": [200, 300]},
    "CatBoost": {"catboostclassifier__depth": [4, 6], "catboostclassifier__l2_leaf_reg": [5, 10, 15], "catboostclassifier__learning_rate": [0.03, 0.05],
                 "catboostclassifier__iterations": [1000]},   # 1 000 arbres : valeur par défaut, réduite par la boucle en cas de surapprentissage
}

# Comment faire évoluer chaque paramètre numérique d'un tour à l'autre (ceux des scénarios) :
# - simplifier : sens qui réduit le surapprentissage (+1 = augmenter, -1 = diminuer) ;
# - pas : écart entre deux valeurs (« x » = multiplier, « + » = ajouter) ;
# - bornes : valeurs minimale et maximale autorisées.
# Les paramètres absents de ce tableau (architecture du MLP, nombre d'arbres du RandomForest) sont fixés à leur meilleure valeur après le premier tour.
REGLAGES_PARAMETRES = {
    "logisticregression__C":                    {"simplifier": -1, "pas": ("x", 3),   "bornes": (1e-4, 1e4),  "entier": False},
    "mlpclassifier__alpha":                     {"simplifier": +1, "pas": ("x", 3),   "bornes": (1e-5, 100),  "entier": False},
    "randomforestclassifier__max_depth":        {"simplifier": -1, "pas": ("+", 2),   "bornes": (2, 30),      "entier": True},
    "randomforestclassifier__min_samples_leaf": {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 2000),    "entier": True},
    "catboostclassifier__depth":                {"simplifier": -1, "pas": ("+", 1),   "bornes": (2, 10),      "entier": True},
    "catboostclassifier__l2_leaf_reg":          {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 200),     "entier": False},
    "catboostclassifier__learning_rate":        {"simplifier": -1, "pas": ("x", 1.5), "bornes": (0.003, 0.3), "entier": False},
    "catboostclassifier__iterations":           {"simplifier": -1, "pas": ("x", 1.5), "bornes": (100, 3000),  "entier": True},
}

def nom_court(parametre):
    return parametre.split("__")[-1]

def arrondir(valeur, entier):
    return int(round(valeur)) if entier else float(f"{valeur:.3g}")

def voisin(valeur, reglage, sens):
    # Valeur suivante dans un sens (+1 = plus grande, -1 = plus petite), dans les bornes
    mode, pas = reglage["pas"]
    if mode == "x":
        nouvelle = valeur * pas if sens > 0 else valeur / pas
    else:
        nouvelle = valeur + pas if sens > 0 else valeur - pas
    bas, haut = reglage["bornes"]
    nouvelle = arrondir(min(max(nouvelle, bas), haut), reglage["entier"])
    return None if nouvelle == valeur else nouvelle

def choisir_index(r):
    # Choix sous contrainte, parmi les réglages qui battent le hasard : le meilleur critère sans surapprentissage ;
    # sinon le moins surappris ; si aucun ne bat le hasard, le meilleur score
    val, std = r[f"mean_test_{CRITERE}"], r[f"std_test_{CRITERE}"]
    ecarts = (r[f"mean_train_{CRITERE}"] - val) / val   # écarts relatifs
    utiles = apprend(val, std)
    sous_seuil = np.where(utiles & (ecarts <= SEUIL_SURAPPRENTISSAGE))[0]
    if len(sous_seuil):
        return int(sous_seuil[np.argmax(val[sous_seuil])])
    if utiles.any():
        idx = np.where(utiles)[0]
        return int(idx[np.argmin(ecarts[idx])])
    return int(np.argmax(val))

def bords_de_grille(grille, meilleurs):
    # Paramètres numériques dont la meilleure valeur est la plus petite ou la plus grande de la grille (+1 = maximum, -1 = minimum)
    bords = {}
    for p, valeurs in grille.items():
        if p in REGLAGES_PARAMETRES and len(valeurs) > 1:
            if meilleurs[p] == max(valeurs):
                bords[p] = +1
            elif meilleurs[p] == min(valeurs):
                bords[p] = -1
    return bords

def grille_suivante(grille, meilleurs, bords, surapprend):
    # Grille du tour suivant et explication des corrections ; None si plus rien ne peut bouger
    nouvelle, actions = {}, []
    for p, valeurs in grille.items():
        v = meilleurs[p]
        reglage = REGLAGES_PARAMETRES.get(p)
        if reglage is None:
            nouvelle[p] = [v]                            # catégorie ou paramètre sans effet : fixé
            continue
        sens = None
        if surapprend and reglage["simplifier"] != 0:
            sens = reglage["simplifier"]                 # pousser vers un modèle plus simple
            raison = "surapprentissage"
        elif p in bords:
            sens = bords[p]                              # prolonger la grille du côté du bord
            raison = "bord de grille"
        if sens is None:
            nouvelle[p] = [v]
            continue
        v1 = voisin(v, reglage, sens)
        v2 = voisin(v1, reglage, sens) if v1 is not None else None
        valeurs_tour = [x for x in (v, v1, v2) if x is not None]
        nouvelle[p] = sorted(set(valeurs_tour))
        if len(nouvelle[p]) > 1:
            actions.append(f"{nom_court(p)} {v} → {nouvelle[p]} ({raison})")
    if not actions:
        return None, ["bornes atteintes : la grille ne peut plus bouger"]
    return nouvelle, actions

def lancer_grille(nom, modele, grille, X):
    grid = GridSearchCV(modele, grille, cv=cv, scoring=scorers, refit=choisir_index, return_train_score=True, n_jobs=-1)
    grid.fit(X, y_train)
    r, i = grid.cv_results_, grid.best_index_
    ligne = ligne_scores(nom,
                         {k: r[f"mean_test_{k}"][i] for k in METRIQUES},
                         r[f"std_test_{CRITERE}"][i],
                         {k: r[f"mean_train_{k}"][i] for k in METRIQUES})
    ligne["Meilleurs paramètres"] = {nom_court(k): v for k, v in grid.best_params_.items()}
    return grid, ligne

def cle_reglage(params):
    # Identifiant d'une combinaison de paramètres, pour savoir si elle a déjà été testée
    return str(sorted(params.items()))

def reglages_nouveaux(grille, deja_testes):
    # Combinaisons de la grille qui n'ont encore été testées à aucun tour
    combinaisons = [dict(zip(grille, valeurs)) for valeurs in product(*grille.values())]
    return [c for c in combinaisons if cle_reglage(c) not in deja_testes]

def choisir_tour(tours):
    # Version retenue parmi les tours joués : d'abord ceux qui battent le hasard et sans surapprentissage
    # (sinon ceux qui battent le hasard, sinon tous) ; parmi eux, ceux à moins d'un écart-type du meilleur score
    # (à égalité avec lui) ; et parmi ces derniers, celui qui surapprend le moins
    for garder in (lambda l: l["Apprend"] and l["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE, lambda l: l["Apprend"], lambda l: True):
        groupe = [t for t in tours if garder(t[1])]
        if groupe:
            break
    meilleur = max(groupe, key=lambda t: t[1]["Critère (val)"])[1]
    egaux = [t for t in groupe if meilleur["Critère (val)"] - t[1]["Critère (val)"] <= meilleur["Écart-type critère (val)"]]
    return min(egaux, key=lambda t: t[1]["Écart relatif"])

def boucle(variante):
    # Boucle des scénarios, pour un jeu de variables : version retenue de chaque modèle, lignes de scores et historique
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    best_models, lignes_retenues, historique = {}, [], []
    for nom, modele in modeles_de_base(variante).items():
        grille, retenu, fin, deja_testes, reglage_precedent, tours = grilles[nom], None, None, set(), None, []
        for tour in range(1, TOURS_SECURITE + 1):
            grid, ligne = lancer_grille(nom, modele, grille, X)
            ligne["Tour"] = tour
            deja_testes |= {cle_reglage(c) for c in grid.cv_results_["params"]}
            surapprend = ligne["Écart relatif"] > SEUIL_SURAPPRENTISSAGE
            bords = bords_de_grille(grille, grid.best_params_)
            precedent = retenu
            tours.append((grid, ligne))
            retenu = choisir_tour(tours)
            grille_tour = {nom_court(p): v for p, v in grille.items()}
            reglage = ligne["Meilleurs paramètres"]
            # Stagnation : seulement entre deux tours sans surapprentissage
            stagne = (precedent is not None and not surapprend
                      and precedent[1]["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE
                      and ligne["Critère (val)"] - precedent[1]["Critère (val)"] <= precedent[1]["Écart-type critère (val)"])
            if not ligne["Apprend"]:
                fin = "sous-apprentissage : le réglage choisi ne bat plus le hasard"
            elif not surapprend and not bords:
                fin = "optimisé : ni surapprentissage ni paramètre au bord"
            elif reglage == reglage_precedent:
                fin = "convergé : le réglage choisi ne change plus"
            elif stagne:
                fin = "stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
            elif tour == TOURS_SECURITE:
                fin = "⚠️ garde-fou atteint : grilles ou bornes à revoir"
            else:
                grille, actions = grille_suivante(grille, grid.best_params_, bords, surapprend)
                if grille is None:
                    fin = actions[0]
                elif not reglages_nouveaux(grille, deja_testes):
                    fin = "convergé : la grille suivante ne contient aucun réglage nouveau"
            reglage_precedent = reglage
            historique.append({
                "Modèle": nom, "Tour": tour, "Grille testée": grille_tour, "Réglage choisi": reglage,
                "Critère (val)": ligne["Critère (val)"], "Écart-type": ligne["Écart-type critère (val)"],
                "Écart relatif": ligne["Écart relatif"], "Alerte": ligne["Alerte"],
                "Suite": fin if fin else "; ".join(actions),
            })
            if fin:
                break
        best_models[nom] = retenu[0].best_estimator_
        lignes_retenues.append(dict(retenu[1], **{"Arrêt": fin, "Tours joués": tour}))
        print(f"  {nom} : {tour} tour(s), {fin} ; version retenue : tour {retenu[1]['Tour']}")
    return best_models, pd.DataFrame(lignes_retenues), pd.DataFrame(historique)

## 3. Boucle d'optimisation, variante par variante

Pour chaque variante, la boucle règle les 4 modèles l'un après l'autre. Une ligne s'affiche pendant le calcul pour chaque modèle (nombre de tours, raison de l'arrêt, tour retenu). Puis, pour chaque variante :
- **Version retenue de chaque modèle** : la précision moyenne en validation (le critère), son écart-type, l'entraînement face à la validation, l'écart relatif (au-delà de 7 % : ⚠️ ; de 14 % : ⚠️⚠️), le ROC AUC en contrôle, et le réglage retenu ;
- **Historique de la boucle** : une ligne par tour et par modèle, avec la grille testée, le réglage choisi et la correction qui suit (ou la raison de l'arrêt).

Le calcul est long : la boucle tourne deux fois, sur 30 000 clients.

In [6]:
import time

resultats = {}
for variante in VARIANTES:
    debut = time.time()
    print(f"Variante « {variante} » ({len(COLONNES_VARIANTE[variante])} variables)")
    resultats[variante] = boucle(variante)
    print(f"  durée : {(time.time() - debut) / 60:.1f} min")

for variante, (_, df_retenus, df_historique) in resultats.items():
    display(Markdown(f"### Variante « {variante} »"))
    df = df_retenus.sort_values("Critère (val)", ascending=False)
    montrer(pd.DataFrame({
        "Modèle": df["Modèle"],
        "Précision moyenne val (%)": pct(df["Critère (val)"]),
        "Écart-type (pts)": pct(df["Écart-type critère (val)"]),
        "Précision moyenne train (%)": pct(df["Critère (train)"]),
        "Écart relatif (%)": pct(df["Écart relatif"]),
        "ROC AUC val": df["ROC AUC (val)"].round(3),
        "ROC AUC train": df["ROC AUC (train)"].round(3),
        "Alerte": df["Alerte"],
        "Tours joués": df["Tours joués"],
        "Tour retenu": df["Tour"],
        "Raison de l'arrêt": df["Arrêt"],
        "Réglage retenu": df["Meilleurs paramètres"].map(parametres),
    }), "Version retenue de chaque modèle (validation)")
    montrer(pd.DataFrame({
        "Modèle": df_historique["Modèle"],
        "Tour": df_historique["Tour"],
        "Grille testée": df_historique["Grille testée"].map(parametres),
        "Réglage choisi": df_historique["Réglage choisi"].map(parametres),
        "Précision moyenne val (%)": pct(df_historique["Critère (val)"]),
        "Écart relatif (%)": pct(df_historique["Écart relatif"]),
        "Alerte": df_historique["Alerte"],
        "Suite": df_historique["Suite"],
    }), "Historique de la boucle")

Variante « Complet » (23 variables)


  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1


  MLPClassifier : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1


  RandomForest : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1


  CatBoost : 3 tour(s), stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage ; version retenue : tour 2
  durée : 7.8 min
Variante « PAY_n seuls » (6 variables)
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1


...

### Variante « PAY_n seuls »

**Version retenue de chaque modèle (validation)**

,Précision moyenne val (%),Écart-type (pts),Précision moyenne train (%),Écart relatif (%),ROC AUC val,ROC AUC train,Alerte,Tours joués,Tour retenu,Raison de l'arrêt,Réglage retenu
Modèle,,,,,,,,,,,
CatBoost,53.2,0.7,54.4,2.1,0.750,0.758,✅ OK,2,2,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage",depth 4 ; iterations 1000 ; l2_leaf_reg 10 ; learning_rate 0.0133
RandomForest,53.2,0.8,54.4,2.2,0.750,0.757,✅ OK,2,1,convergé : le réglage choisi ne change plus,max_depth 8 ; min_samples_leaf 20 ; n_estimators 200
MLPClassifier,51.8,0.5,52.2,0.8,0.743,0.745,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,"alpha 0.1 ; hidden_layer_sizes (64, 32)"
LogisticRegression,48.6,0.8,48.6,-0.1,0.705,0.705,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,C 0.1


**Historique de la boucle**

,Tour,Grille testée,Réglage choisi,Précision moyenne val (%),Écart relatif (%),Alerte,Suite
Modèle,,,,,,,
LogisticRegression,1,"C 0.01, 0.1, 1.0, 10.0",C 0.1,48.6,-0.1,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
MLPClassifier,1,"hidden_layer_sizes (32, 16), (64, 32), (32,) ; alpha 0.01, 0.1, 1.0","alpha 0.1 ; hidden_layer_sizes (64, 32)",51.8,0.8,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
RandomForest,1,"max_depth 6, 8, 10 ; min_samples_leaf 20, 50, 100 ; n_estimators 200, 300",max_depth 8 ; min_samples_leaf 20 ; n_estimators 200,53.2,2.2,✅ OK,"min_samples_leaf 20 → [9, 13, 20] (bord de grille)"
RandomForest,2,"max_depth 8 ; min_samples_leaf 9, 13, 20 ; n_estimators 200",max_depth 8 ; min_samples_leaf 20 ; n_estimators 200,53.2,2.2,✅ OK,convergé : le réglage choisi ne change plus
CatBoost,1,"depth 4, 6 ; l2_leaf_reg 5, 10, 15 ; learning_rate 0.03, 0.05 ; iterations 1000",depth 4 ; iterations 1000 ; l2_leaf_reg 10 ; learning_rate 0.03,53.1,3.9,✅ OK,"depth 4 → [2, 3, 4] (bord de grille); learning_rate 0.03 → [0.0133, 0.02, 0.03] (bord de grille)"
CatBoost,2,"depth 2, 3, 4 ; l2_leaf_reg 10 ; learning_rate 0.0133, 0.02, 0.03 ; iterations 1000",depth 4 ; iterations 1000 ; l2_leaf_reg 10 ; learning_rate 0.0133,53.2,2.1,✅ OK,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"


**Lecture** : une section par variante. Le taux de défaut du jeu de base est la précision moyenne d'un tri au hasard : une précision moyenne proche de lui veut dire que le modèle ne trouve presque rien. Un modèle qui reste en surapprentissage malgré tous ses tours est signalé (⚠️) : sa grille de départ ou ses bornes sont peut-être à revoir pour ces données.

## 4. Seuil de décision optimisé et comparaison des variantes, pli par pli

**Seuil optimisé automatiquement** : sur chaque pli, la version retenue de chaque modèle est réentraînée sur les 4 autres plis, enveloppée dans `TunedThresholdClassifierCV`. Celui-ci découpe à son tour ces 4 plis en 5 plis internes et essaie une série de seuils sur les prédictions internes. Il garde celui qui maximise le **gain net** « défauts détectés − `BONS_CLIENTS_PAR_DEFAUT` × bons clients signalés », puis réentraîne le modèle sur les 4 plis. Le pli de validation n'intervient ni dans les réglages du modèle ni dans le choix du seuil : il juge seulement le résultat. Chaque client reçoit une probabilité hors pli et une décision (signalé ou non) prise au seuil de son pli. Les modèles entraînés sur chaque pli sont gardés en mémoire pour la section 5.

**Contrôle de la règle** : la précision **marginale** se mesure sur les clients signalés les plus proches du seuil (les 10 % de clients signalés qui ont les probabilités les plus basses). Elle doit tourner autour de 1 / (1 + `BONS_CLIENTS_PAR_DEFAUT`), soit 50 % : c'est la part de défauts parmi les derniers clients que la règle a jugé bon d'ajouter.

Trois tableaux :
- **Détection au seuil optimisé** : pour chaque modèle et chaque variante, le seuil choisi (moyenne et étendue sur les 5 plis), le **rappel** (part des défauts détectés), la **précision des défauts prédits** (sur tous les clients signalés), la **précision marginale** (sur les 10 % de signalés les plus proches du seuil), la part des clients signalés et le F1 (pour information), en moyenne sur les plis avec leur écart-type ; puis la précision moyenne et le ROC AUC, qui ne dépendent d'aucun seuil ;
- **Comparaison avec la variante complète** : pour chaque modèle, puis en meilleur contre meilleur, le gain moyen sur la variante complète, son écart-type et les plis gagnés, pour le rappel au seuil optimisé (combien de défauts sont détectés avec la règle) et pour la précision moyenne (la capacité de détection, tous seuils confondus). Verdict des scénarios : « gain réel » si le gain moyen dépasse l'écart-type des gains, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [7]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, fbeta_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TunedThresholdClassifierCV

y = y_train.to_numpy()
plis = list(cv.split(original, y_train))   # les mêmes 5 plis que la boucle
cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # plis internes, pour le choix du seuil

# Règle fixée à l'avance par le métier : au plus 1 bon client signalé pour 1 défaut attrapé, à la marge
BONS_CLIENTS_PAR_DEFAUT = 1
PRECISION_MARGINALE_VISEE = 1 / (1 + BONS_CLIENTS_PAR_DEFAUT)   # 50 % : précision des derniers clients ajoutés au seuil retenu
PART_MARGE = 0.10   # contrôle : précision mesurée sur les 10 % de clients signalés les plus proches du seuil

def gain_net(y_vrai, y_pred):
    # Défauts détectés moins les bons clients signalés (pondérés), rapporté au nombre de clients :
    # maximal au seuil où les clients ajoutés ne sont plus en défaut qu'avec la précision marginale visée
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    detectes = (y_pred & (y_vrai == 1)).sum()
    fausses_alertes = (y_pred & (y_vrai == 0)).sum()
    return (detectes - BONS_CLIENTS_PAR_DEFAUT * fausses_alertes) / len(y_vrai)

SCORE_SEUIL = make_scorer(gain_net)

def precision_marginale(y_vrai, proba, signales):
    # Part de défauts parmi les PART_MARGE des clients signalés qui ont les probabilités les plus basses (les plus proches du seuil)
    idx = np.where(signales)[0]
    if len(idx) == 0:
        return np.nan
    derniers = idx[np.argsort(proba[idx])[:max(1, int(np.ceil(PART_MARGE * len(idx))))]]
    return y_vrai[derniers].mean()

def f2_au_rappel_minimal(y_vrai, proba):
    # Mesure des scénarios (section 6) : F2 au seuil qui détecte au moins le rappel minimal
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

probas, mesures, modeles_plis = {}, {}, {}
for variante, (best_models, _, _) in resultats.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for nom, modele in best_models.items():
        p, entraines = np.zeros(len(y)), []
        m_plis = {"Seuil": [], "Rappel": [], "Précision": [], "Précision marginale": [], "Signalés": [], "F1": [], "Précision moyenne": [], "ROC AUC": []}
        for entrainement, validation in plis:
            seuil_auto = TunedThresholdClassifierCV(clone(modele), scoring=SCORE_SEUIL, cv=cv_interne, n_jobs=-1)
            seuil_auto.fit(X.iloc[entrainement], y[entrainement])
            p[validation] = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
            decision = seuil_auto.predict(X.iloc[validation]).astype(bool)
            y_val = y[validation]
            m_plis["Seuil"].append(seuil_auto.best_threshold_)
            m_plis["Rappel"].append(recall_score(y_val, decision))
            m_plis["Précision"].append(precision_score(y_val, decision, zero_division=0))
            m_plis["Précision marginale"].append(precision_marginale(y_val, p[validation], decision))
            m_plis["Signalés"].append(decision.mean())
            m_plis["F1"].append(fbeta_score(y_val, decision, beta=1))
            m_plis["Précision moyenne"].append(average_precision_score(y_val, p[validation]))
            m_plis["ROC AUC"].append(roc_auc_score(y_val, p[validation]))
            entraines.append(seuil_auto)
        probas[(variante, nom)], mesures[(variante, nom)], modeles_plis[(variante, nom)] = p, m_plis, entraines
        print(f"{variante}, {nom} : seuil {np.mean(m_plis['Seuil']):.3f}, rappel {np.mean(m_plis['Rappel']) * 100:.1f} %, "
              f"précision {np.mean(m_plis['Précision']) * 100:.1f} %, précision marginale {np.nanmean(m_plis['Précision marginale']) * 100:.1f} %")

def moy_et(valeurs, en_pct=False):
    facteur = 100 if en_pct else 1
    return f"{np.mean(valeurs) * facteur:.{1 if en_pct else 3}f} ± {np.std(valeurs) * facteur:.{1 if en_pct else 3}f}"

lignes = []
for (variante, nom), m in mesures.items():
    lignes.append({
        "Variante": variante, "Modèle": nom,
        "Seuil (moyenne)": round(np.mean(m["Seuil"]), 3),
        "Seuil (min - max)": f"{min(m['Seuil']):.3f} - {max(m['Seuil']):.3f}",
        "Rappel (%)": moy_et(m["Rappel"], True),
        f"{LIBELLE_SCORE} (%)": moy_et(m["Précision"], True),
        f"Précision marginale, {PART_MARGE:.0%} les plus proches du seuil (%)": moy_et(m["Précision marginale"], True),
        "Clients signalés (%)": moy_et(m["Signalés"], True),
        "F1": moy_et(m["F1"]),
        "Précision moyenne": moy_et(m["Précision moyenne"]),
        "ROC AUC": moy_et(m["ROC AUC"]),
    })
display(Markdown(f"**Détection au seuil optimisé (au plus {BONS_CLIENTS_PAR_DEFAUT} bon(s) client(s) pour 1 défaut à la marge, "
                 f"précision marginale visée {PRECISION_MARGINALE_VISEE:.0%}), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : {TAUX_DEFAUT * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

Complet, LogisticRegression : seuil 0.753, rappel 28.7 %, précision 63.1 %, précision marginale 51.7 %


Complet, MLPClassifier : seuil 0.508, rappel 34.9 %, précision 67.6 %, précision marginale 57.1 %


Complet, RandomForest : seuil 0.725, rappel 36.8 %, précision 64.8 %, précision marginale 51.9 %


Complet, CatBoost : seuil 0.763, rappel 37.8 %, précision 66.5 %, précision marginale 51.4 %


PAY_n seuls, LogisticRegression : seuil 0.690, rappel 37.1 %, précision 67.0 %, précision marginale 57.7 %


PAY_n seuls, MLPClassifier : seuil 0.500, rappel 36.6 %, précision 67.4 %, précision marginale 52.0 %


PAY_n seuls, RandomForest : seuil 0.782, rappel 35.5 %, précision 68.4 %, précision marginale 57.6 %


PAY_n seuls, CatBoost : seuil 0.792, rappel 35.4 %, précision 68.1 %, précision marginale 53.2 %


**Détection au seuil optimisé (au plus 1 bon(s) client(s) pour 1 défaut à la marge, précision marginale visée 50%), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : 22.1 %)

Seuil (moyenne) Seuil (min - max)  Rappel (%) Précision des défauts prédits (%) Précision marginale, 10% les plus proches du seuil (%) Clients signalés (%)             F1 Précision moyenne        ROC AUC
Variante    Modèle                                                                                                                                                                                                                         
Complet     LogisticRegression            0.753     0.743 - 0.763  28.7 ± 1.9                        63.1 ± 0.8                                             51.7 ± 3.6           10.1 ± 0.7  0.394 ± 0.017     0.498 ± 0.005  0.749 ± 0.004
            MLPClassifier                 0.508     0.481 - 0.558  34.9 ± 2.3                        67.6 ± 1.2                                             57.1 ± 7.1           11.4 ± 0.8  0.460 ± 0.020     0.540 ± 0.006  0.777 ± 0.004
            RandomForest                  0.725     0.710 - 0.746  36.8 ± 2.2                        64.8 ± 1.0                                             51.9 ± 6.1           12.6 ± 0.8  0.469 ± 0.017     0.551 ± 0.008  0.780 ± 0.006
            CatBoost                      0.763     0.746 - 0.779  37.8 ± 1.3                        66.5 ± 1.3                                             51.4 ± 4.9           12.6 ± 0.5  0.482 ± 0.011     0.556 ± 0.007  0.784 ± 0.005
PAY_n seuls LogisticRegression            0.690     0.685 - 0.701  37.1 ± 1.6                        67.0 ± 2.3                                             57.7 ± 9.9           12.3 ± 0.7  0.477 ± 0.013     0.486 ± 0.008  0.705 ± 0.006
            MLPClassifier                 0.500     0.463 - 0.564  36.6 ± 3.1                        67.4 ± 2.5                                             52.0 ± 5.0           12.1 ± 1.4  0.473 ± 0.023     0.518 ± 0.005  0.743 ± 0.011
            RandomForest                  0.782     0.775 - 0.785  35.5 ± 1.4                        68.4 ± 1.8                                             57.6 ± 3.7           11.5 ± 0.2  0.468 ± 0.016     0.532 ± 0.008  0.750 ± 0.006
            CatBoost                      0.792     0.785 - 0.798  35.4 ± 0.7                        68.1 ± 2.0                                             53.2 ± 7.0           11.5 ± 0.2  0.466 ± 0.010     0.532 ± 0.007  0.750 ± 0.006

In [8]:
def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

REFERENCE = "Complet"
for mesure in ("Rappel", "Précision moyenne"):
    libelle = "Rappel au seuil optimisé" if mesure == "Rappel" else mesure
    def meilleur(variante):
        return max(MODELES_A_TESTER, key=lambda nom: np.mean(mesures[(variante, nom)][mesure]))
    lignes = []
    for variante in [v for v in VARIANTES if v != REFERENCE]:
        for nom in MODELES_A_TESTER + ["Meilleur contre meilleur"]:
            ref, comp = (meilleur(REFERENCE), meilleur(variante)) if nom == "Meilleur contre meilleur" else (nom, nom)
            score_ref, score_comp = np.array(mesures[(REFERENCE, ref)][mesure]), np.array(mesures[(variante, comp)][mesure])
            gains = score_comp - score_ref
            lignes.append({
                "Variante": variante,
                "Modèle": nom if nom != "Meilleur contre meilleur" else f"{nom} ({ref} → {comp})",
                f"{libelle}, complet": round(score_ref.mean(), 3),
                f"{libelle}, variante": round(score_comp.mean(), 3),
                "Gain": f"{gains.mean():+.3f} ± {gains.std():.3f}",
                "Part du score complet gardée (%)": round(score_comp.mean() / score_ref.mean() * 100, 1),
                "Plis gagnés": f"{(gains > 0).sum()} sur 5",
                "Verdict": verdict(gains),
            })
    display(Markdown(f"**Comparaison avec la variante complète, pli par pli : {libelle}**"))
    display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

**Comparaison avec la variante complète, pli par pli : Rappel au seuil optimisé**

Rappel au seuil optimisé, complet  Rappel au seuil optimisé, variante            Gain  Part du score complet gardée (%) Plis gagnés          Verdict
Variante    Modèle                                                                                                                                                                                                        
PAY_n seuls LogisticRegression                                                                    0.287                               0.371  +0.084 ± 0.033                             129.2     5 sur 5      ✅ gain réel
            MLPClassifier                                                                         0.349                               0.366  +0.017 ± 0.030                             104.8     3 sur 5  ➖ dans le bruit
            RandomForest                                                                          0.368                               0.355  -0.012 ± 0.014                              96.6     1 sur 5  ➖ dans le bruit
            CatBoost                                                                              0.378                               0.354  -0.024 ± 0.013                              93.6     0 sur 5          ❌ perte
            Meilleur contre meilleur (CatBoost → LogisticRegression)                              0.378                               0.371  -0.007 ± 0.012                              98.1     1 sur 5  ➖ dans le bruit

**Comparaison avec la variante complète, pli par pli : Précision moyenne**

Précision moyenne, complet  Précision moyenne, variante            Gain  Part du score complet gardée (%) Plis gagnés  Verdict
Variante    Modèle                                                                                                                                                                        
PAY_n seuls LogisticRegression                                                   0.498                        0.486  -0.012 ± 0.005                              97.6     0 sur 5  ❌ perte
            MLPClassifier                                                        0.540                        0.518  -0.023 ± 0.009                              95.8     0 sur 5  ❌ perte
            RandomForest                                                         0.551                        0.532  -0.019 ± 0.004                              96.6     0 sur 5  ❌ perte
            CatBoost                                                             0.556                        0.532  -0.023 ± 0.003                              95.8     0 sur 5  ❌ perte
            Meilleur contre meilleur (CatBoost → CatBoost)                       0.556                        0.532  -0.023 ± 0.003                              95.8     0 sur 5  ❌ perte

**Comment lire** : comme dans `essai_jeu_de_base`. Le rappel au seuil optimisé dit combien de défauts chaque modèle détecte avec la règle marginale (au plus 1 bon client pour 1 défaut) ; la précision marginale, proche de 50 %, vérifie que le seuil s'arrête au bon endroit. La comparaison « `PAY_n` seuls » face à « Complet » se lit ici avec des codifications corrigées : elle se compare à la même ligne d'`essai_jeu_de_base`.

In [9]:
# Enregistrement, client par client, des probabilités hors pli et des décisions au seuil optimisé (règle marginale),
# pour chaque variante et chaque modèle, dans le dossier local sorties/. Aucun réentraînement :
# les décisions sont celles des modèles de chaque pli, gardés en mémoire à la section 4
NOM_ESSAI = "jeu_de_base_correctifs"
dossier_sortie = Path.cwd() / "sorties"
dossier_sortie.mkdir(parents=True, exist_ok=True)
sortie = pd.DataFrame({"ID": original["ID"].to_numpy(), "dpnm": y})
for (variante, nom), p in probas.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    decision = np.zeros(len(y), dtype=int)
    for m, (_, validation) in zip(modeles_plis[(variante, nom)], plis):
        decision[validation] = m.predict(X.iloc[validation])
    sortie[f"proba | {variante} | {nom}"] = p
    sortie[f"decision | {variante} | {nom}"] = decision
sortie.to_csv(dossier_sortie / f"{NOM_ESSAI}.csv", index=False)
print(f"Enregistré : {dossier_sortie / f'{NOM_ESSAI}.csv'} ({len(sortie)} clients, {len(probas)} couples variante × modèle)")

Enregistré : C:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\sorties\jeu_de_base_correctifs.csv (30000 clients, 8 couples variante × modèle)


## 5. Ce qui porte les prédictions : importance par permutation

**Principe** : sur chaque pli de validation, on reprend le modèle entraîné sur les 4 autres plis (section 4) et on **mélange** les valeurs d'une famille de colonnes entre les clients (les autres colonnes restent intactes). Le modèle perd ce que cette famille lui apportait : la **perte de précision moyenne** mesure son importance. La précision moyenne ne dépend d'aucun seuil : la mesure juge la capacité de détection, pas un point de fonctionnement, et elle est plus stable qu'un score pris en un seul point. Le mélange est répété 5 fois (graine fixée), puis la perte est moyennée sur les répétitions et sur les 5 plis. Une famille se mélange d'un bloc (les six mois ensemble, avec le même tirage pour chaque colonne), pour garder sa cohérence d'un mois à l'autre.

**Familles** : codifications (`PAY_1` à `PAY_6`), factures (`BILL_AMT1` à `6`), paiements (`PAY_AMT1` à `6`), plafond (`LIMIT_BAL`), démographie (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`). Le comportement du client, au sens des montants, ce sont les factures, les paiements et le plafond ; les codifications sont l'étiquette posée par la banque.

**Deux tableaux** :
- **par famille**, pour chaque variante et chaque modèle : perte de précision moyenne et son écart-type sur les plis, et la part de chaque famille dans la perte totale de la colonne ;
- **par colonne**, mélangée seule : le détail, pour voir quels mois et quels montants comptent.

**Ce que la permutation mesure** : ce que le modèle perd **quand la famille manque, les autres étant là**. Deux familles qui disent la même chose (les codifications et les paiements, par exemple) se remplacent l'une l'autre : chacune paraît alors peu importante, alors qu'ensemble elles portent beaucoup. C'est pourquoi cette section se lit avec la section 4 : les variantes « `PAY_n` seuls » et « Sans les `PAY_n` » disent ce que chaque bloc porte **seul**, la permutation dit ce qu'il apporte **en plus des autres**.

In [10]:
FAMILLES = {
    "Codifications (PAY_n)": PAY,
    "Factures (BILL_AMTn)": [f"BILL_AMT{n}" for n in range(1, 7)],
    "Paiements (PAY_AMTn)": [f"PAY_AMT{n}" for n in range(1, 7)],
    "Plafond (LIMIT_BAL)": ["LIMIT_BAL"],
    "Démographie": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
REPETITIONS = 5
generateur = np.random.default_rng(42)

def perte_par_permutation(modeles, X, colonnes):
    # Perte de précision moyenne sur chaque pli de validation quand les colonnes sont mélangées d'un bloc (même tirage pour toutes),
    # moyennée sur les répétitions
    pertes = []
    for m, (_, validation) in zip(modeles, plis):
        X_val, y_val = X.iloc[validation], y[validation]
        reference = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        perte = []
        for _ in range(REPETITIONS):
            melange = X_val.copy()
            ordre_melange = generateur.permutation(len(melange))
            melange[colonnes] = melange[colonnes].to_numpy()[ordre_melange]
            perte.append(reference - average_precision_score(y_val, m.predict_proba(melange)[:, 1]))
        pertes.append(np.mean(perte))
    return pertes

importance_familles, importance_colonnes = {}, {}
for (variante, nom), modeles in modeles_plis.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for famille, colonnes in FAMILLES.items():
        presentes = [c for c in colonnes if c in X.columns]
        if presentes:
            importance_familles[(variante, nom, famille)] = perte_par_permutation(modeles, X, presentes)
    for colonne in X.columns:
        importance_colonnes[(variante, nom, colonne)] = perte_par_permutation(modeles, X, [colonne])
    print(f"{variante}, {nom} : fait")

Complet, LogisticRegression : fait


Complet, MLPClassifier : fait


Complet, RandomForest : fait


Complet, CatBoost : fait


PAY_n seuls, LogisticRegression : fait


PAY_n seuls, MLPClassifier : fait


PAY_n seuls, RandomForest : fait


PAY_n seuls, CatBoost : fait


In [11]:
# Par famille : une colonne par variante et par modèle, une ligne par famille
colonnes_tableau = list(dict.fromkeys((variante, nom) for variante, nom, _ in importance_familles))
perte = pd.DataFrame({cle: {famille: np.mean(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
ecart = pd.DataFrame({cle: {famille: np.std(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
part = perte.clip(lower=0) / perte.clip(lower=0).sum() * 100

display(Markdown("**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)"))
texte = (perte.round(3).astype(str) + " ± " + ecart.round(3).astype(str)).where(perte.notna(), "—")
display(texte.rename_axis(index="Famille", columns=["Variante", "Modèle"]))
display(Markdown("**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)"))
display(part.round(1).rename_axis(index="Famille", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)

Variante                         Complet                                                     PAY_n seuls                                            
Modèle                LogisticRegression  MLPClassifier   RandomForest       CatBoost LogisticRegression  MLPClassifier  RandomForest       CatBoost
Famille                                                                                                                                             
Codifications (PAY_n)       0.22 ± 0.007  0.271 ± 0.012  0.292 ± 0.007   0.29 ± 0.008      0.265 ± 0.008  0.295 ± 0.005  0.31 ± 0.009  0.313 ± 0.006
Factures (BILL_AMTn)         0.0 ± 0.001  0.029 ± 0.004  0.015 ± 0.003  0.019 ± 0.003                  —              —             —              —
Paiements (PAY_AMTn)       0.071 ± 0.005   0.04 ± 0.004  0.027 ± 0.003  0.027 ± 0.004                  —              —             —              —
Plafond (LIMIT_BAL)        0.004 ± 0.002  0.008 ± 0.003  0.006 ± 0.001  0.013 ± 0.002                  —              —             —              —
Démographie                0.007 ± 0.001  0.006 ± 0.003  0.001 ± 0.001  0.006 ± 0.002                  —              —             —              —

**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)

Variante                         Complet                                            PAY_n seuls                                    
Modèle                LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Famille                                                                                                                            
Codifications (PAY_n)               72.7          76.6         85.8     82.1              100.0         100.0        100.0    100.0
Factures (BILL_AMTn)                 0.0           8.1          4.4      5.3                NaN           NaN          NaN      NaN
Paiements (PAY_AMTn)                23.5          11.2          7.8      7.5                NaN           NaN          NaN      NaN
Plafond (LIMIT_BAL)                  1.3           2.3          1.7      3.6                NaN           NaN          NaN      NaN
Démographie                          2.4           1.7          0.3      1.6                NaN           NaN          NaN      NaN

In [12]:
# Par colonne, mélangée seule : perte de précision moyenne sur les 5 plis, triée sur la variante complète (moyenne des 4 modèles)
par_colonne = pd.DataFrame({(variante, nom): {colonne: np.mean(importance_colonnes[(variante, nom, colonne)])
                                              for v, n, colonne in importance_colonnes if (v, n) == (variante, nom)}
                            for variante, nom in colonnes_tableau})
ordre_colonnes = par_colonne["Complet"].mean(axis=1).sort_values(ascending=False).index
with pd.option_context("display.max_rows", None):
    display(Markdown("**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)"))
    display(par_colonne.loc[ordre_colonnes].round(3).rename_axis(index="Colonne", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)

Variante             Complet                                            PAY_n seuls                                    
Modèle    LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Colonne                                                                                                                
PAY_1                  0.105         0.215        0.137    0.191              0.219         0.204        0.186    0.199
PAY_2                  0.017         0.007        0.013    0.005              0.001         0.016        0.023    0.016
LIMIT_BAL              0.004         0.008        0.006    0.012                NaN           NaN          NaN      NaN
PAY_6                  0.011         0.009        0.005    0.005             -0.006         0.012        0.007    0.010
BILL_AMT1             -0.003         0.014        0.004    0.009                NaN           NaN          NaN      NaN
PAY_3                  0.007         0.005        0.006    0.005             -0.001         0.008        0.007    0.008
PAY_4                  0.005         0.005        0.008    0.005              0.001         0.014        0.010    0.008
PAY_5                  0.006         0.009        0.004    0.002             -0.004         0.004        0.002    0.003
PAY_AMT2               0.005         0.008        0.003    0.003                NaN           NaN          NaN      NaN
PAY_AMT1               0.001         0.011        0.003    0.002                NaN           NaN          NaN      NaN
BILL_AMT3              0.008         0.003        0.001    0.001                NaN           NaN          NaN      NaN
PAY_AMT3               0.001         0.004        0.002    0.002                NaN           NaN          NaN      NaN
BILL_AMT2             -0.001         0.008        0.002    0.001                NaN           NaN          NaN      NaN
EDUCATION              0.003         0.002        0.000    0.003                NaN           NaN          NaN      NaN
BILL_AMT6              0.005         0.002        0.001    0.001                NaN           NaN          NaN      NaN
BILL_AMT4              0.003         0.004        0.001    0.001                NaN           NaN          NaN      NaN
MARRIAGE               0.002         0.002        0.000    0.001                NaN           NaN          NaN      NaN
BILL_AMT5             -0.001         0.005        0.001    0.001                NaN           NaN          NaN      NaN
PAY_AMT4              -0.001         0.002        0.002    0.002                NaN           NaN          NaN      NaN
PAY_AMT6              -0.001         0.001        0.001    0.002                NaN           NaN          NaN      NaN
SEX                    0.001         0.001        0.000    0.001                NaN           NaN          NaN      NaN
PAY_AMT5              -0.000         0.002        0.001    0.000                NaN           NaN          NaN      NaN
AGE                    0.000         0.001        0.000    0.001                NaN           NaN          NaN      NaN

**Comment lire** :
- **Par famille** : plus la perte est grande, plus le modèle s'appuie sur la famille. Une perte plus petite que son écart-type est dans le bruit : la famille n'apporte rien de plus que les autres. Si, dans la variante complète, les codifications portent l'essentiel de la perte, les modèles s'appuient d'abord sur l'étiquette de la banque ; si ce sont les factures et les paiements, ils s'appuient sur le comportement de paiement du client.
- **Par colonne** : une colonne mélangée seule perd moins que sa famille, puisque les autres mois la remplacent en partie. Le tableau sert surtout à voir **quel mois** compte (`PAY_1`, le plus récent, ou l'historique) et **quel montant**.
- **Avec la section 4** : une famille peu importante ici, mais qui garde l'essentiel du score seule (variante « `PAY_n` seuls ») ou sans laquelle le score tient (variante « Sans les `PAY_n` »), dit la même chose que les autres familles : l'information est **redondante**, pas absente.

## 6. Comparaison avec `essai_jeu_de_base` : précision à rappel égal, pli par pli

**Ce qui est comparé** : pour chaque variante (« Complet », « `PAY_n` seuls ») et chaque modèle, les probabilités hors pli de cet essai face à celles d'`essai_jeu_de_base` (`data/ML/comparatif_global/jeu_de_base.csv`). Mêmes clients, même ordre, mêmes 5 plis : chaque client est jugé par un modèle qui ne l'a pas vu, dans les deux essais. Seules les codifications d'avril à août diffèrent.

**Mesures, sur chaque pli** :
- **précision à rappel égal** : on signale, du plus risqué au moins risqué, juste assez de clients pour détecter 50 %, 60 %, 70 % puis 80 % des défauts du pli, et on mesure la part de défauts parmi les clients signalés ;
- **précision moyenne** : la même chose résumée sur tous les rappels ;
- **à la règle marginale** (au plus 1 bon client pour 1 défaut) : rappel et précision au seuil optimisé de chaque essai.

Verdict d'`essai_jeu_de_base` : « gain réel » si le gain moyen des correctifs dépasse l'écart-type des gains sur les plis, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [13]:
reference = pd.read_csv(root_dir / "data" / "ML" / "comparatif_global" / "jeu_de_base.csv")
assert (reference["ID"].to_numpy() == original["ID"].to_numpy()).all(), "clients pas dans le même ordre"
assert (reference["dpnm"].to_numpy() == y).all(), "cible différente"

RAPPELS = [0.50, 0.60, 0.70, 0.80]

def precision_au_rappel(y_vrai, proba, rappel):
    # Part de défauts parmi les clients signalés au seuil le plus haut qui détecte au moins la part voulue des défauts
    return y_vrai[proba >= seuil_pour_rappel(y_vrai, proba, rappel)].mean()

def mesures_plis(proba, decision):
    m = {f"Précision au rappel de {r:.0%}": [] for r in RAPPELS}
    m.update({"Précision moyenne": [], "Rappel, règle marginale": [], "Précision, règle marginale": []})
    for _, validation in plis:
        y_val, p_val, d_val = y[validation], proba[validation], decision[validation].astype(bool)
        for r in RAPPELS:
            m[f"Précision au rappel de {r:.0%}"].append(precision_au_rappel(y_val, p_val, r))
        m["Précision moyenne"].append(average_precision_score(y_val, p_val))
        m["Rappel, règle marginale"].append(recall_score(y_val, d_val))
        m["Précision, règle marginale"].append(precision_score(y_val, d_val, zero_division=0))
    return {k: np.array(v) for k, v in m.items()}

decisions = {}
for (variante, nom), p in probas.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    d = np.zeros(len(y), dtype=int)
    for m, (_, validation) in zip(modeles_plis[(variante, nom)], plis):
        d[validation] = m.predict(X.iloc[validation])
    decisions[(variante, nom)] = d

comparaison = {}
for (variante, nom), p in probas.items():
    comparaison[(variante, nom)] = {
        "jeu de base": mesures_plis(reference[f"proba | {variante} | {nom}"].to_numpy(), reference[f"decision | {variante} | {nom}"].to_numpy()),
        "correctifs": mesures_plis(p, decisions[(variante, nom)]),
    }
MESURES_COMPAREES = list(next(iter(comparaison.values()))["correctifs"])

def ligne_comparaison(variante, libelle, ref, comp, mesure):
    gains = comp[mesure] - ref[mesure]
    return {"Variante": variante, "Modèle": libelle, "Mesure": mesure,
            "Jeu de base (%)": round(ref[mesure].mean() * 100, 1), "Avec correctifs (%)": round(comp[mesure].mean() * 100, 1),
            "Gain (pts)": f"{gains.mean() * 100:+.2f} ± {gains.std() * 100:.2f}",
            "Plis gagnés": f"{(gains > 0).sum()} sur 5", "Verdict": verdict(gains)}

lignes = []
for variante in VARIANTES:
    for nom in MODELES_A_TESTER:
        c = comparaison[(variante, nom)]
        for mesure in MESURES_COMPAREES:
            lignes.append(ligne_comparaison(variante, nom, c["jeu de base"], c["correctifs"], mesure))
    # Meilleur contre meilleur : le meilleur modèle de chaque essai, sur la précision moyenne
    meilleur_ref = max(MODELES_A_TESTER, key=lambda n: comparaison[(variante, n)]["jeu de base"]["Précision moyenne"].mean())
    meilleur_comp = max(MODELES_A_TESTER, key=lambda n: comparaison[(variante, n)]["correctifs"]["Précision moyenne"].mean())
    for mesure in MESURES_COMPAREES:
        lignes.append(ligne_comparaison(variante, f"Meilleur contre meilleur ({meilleur_ref} → {meilleur_comp})",
                                        comparaison[(variante, meilleur_ref)]["jeu de base"], comparaison[(variante, meilleur_comp)]["correctifs"], mesure))
table = pd.DataFrame(lignes)

display(Markdown("**Synthèse : gain des correctifs, en points, par mesure** (moyenne sur les 5 plis ± écart-type ; une colonne par modèle)"))
display(table.pivot_table(index=["Variante", "Mesure"], columns="Modèle", values="Gain (pts)", aggfunc="first", sort=False))
display(Markdown("**Verdicts, par mesure**"))
display(table.pivot_table(index=["Variante", "Mesure"], columns="Modèle", values="Verdict", aggfunc="first", sort=False))
with pd.option_context("display.max_rows", None):
    display(Markdown("**Détail, pli par pli : jeu de base face au jeu avec correctifs**"))
    display(table.set_index(["Variante", "Modèle", "Mesure"]))

**Synthèse : gain des correctifs, en points, par mesure** (moyenne sur les 5 plis ± écart-type ; une colonne par modèle)

Modèle                                 LogisticRegression MLPClassifier  RandomForest      CatBoost Meilleur contre meilleur (CatBoost → CatBoost) Meilleur contre meilleur (RandomForest → CatBoost)
Variante    Mesure                                                                                                                                                                                   
Complet     Précision au rappel de 50%       -0.29 ± 0.29  -0.76 ± 0.99  +0.02 ± 0.36  -0.12 ± 0.13                                   -0.12 ± 0.13                                                NaN
            Précision au rappel de 60%       -0.01 ± 0.25  +0.51 ± 0.58  +0.54 ± 0.12  +0.11 ± 1.00                                   +0.11 ± 1.00                                                NaN
            Précision au rappel de 70%       +0.03 ± 0.30  +0.02 ± 0.27  -0.14 ± 0.40  -0.25 ± 0.63                                   -0.25 ± 0.63                                                NaN
            Précision au rappel de 80%       +0.18 ± 0.16  +0.54 ± 0.50  -0.06 ± 0.34  +0.26 ± 0.25                                   +0.26 ± 0.25                                                NaN
            Précision moyenne                -0.13 ± 0.09  +0.01 ± 0.14  -0.03 ± 0.18  -0.06 ± 0.11                                   -0.06 ± 0.11                                                NaN
            Rappel, règle marginale          -1.28 ± 1.18  -0.59 ± 3.28  -1.33 ± 2.90  +0.54 ± 0.84                                   +0.54 ± 0.84                                                NaN
            Précision, règle marginale       +0.61 ± 0.67  +1.20 ± 2.06  +0.91 ± 1.39  -0.13 ± 0.73                                   -0.13 ± 0.73                                                NaN
PAY_n seuls Précision au rappel de 50%       +0.25 ± 0.85  -0.67 ± 0.61  +0.77 ± 0.39  +0.04 ± 0.32                                            NaN                                       +0.90 ± 1.07
            Précision au rappel de 60%       +3.16 ± 6.65  +0.49 ± 0.40  +0.30 ± 0.73  +0.69 ± 0.87                                            NaN                                       +0.41 ± 0.71
            Précision au rappel de 70%       -0.01 ± 0.05  +0.11 ± 4.12  -0.19 ± 1.90  -0.89 ± 0.57                                            NaN                                       -0.68 ± 2.15
            Précision au rappel de 80%       -0.15 ± 0.08  -0.04 ± 0.84  +0.18 ± 0.24  +0.08 ± 0.34                                            NaN                                       -0.17 ± 0.36
            Précision moyenne                +0.02 ± 0.10  +0.01 ± 0.21  +0.17 ± 0.14  +0.24 ± 0.32                                            NaN                                       +0.19 ± 0.30
            Rappel, règle marginale          -0.71 ± 1.31  +1.25 ± 2.47  +0.05 ± 0.34  -0.86 ± 0.85                                            NaN                                       -0.12 ± 0.57
            Précision, règle marginale       +0.39 ± 0.99  -0.66 ± 1.91  -0.14 ± 0.26  +0.75 ± 0.59                                            NaN                                       -0.46 ± 0.30

**Verdicts, par mesure**

Modèle                                 LogisticRegression    MLPClassifier     RandomForest         CatBoost Meilleur contre meilleur (CatBoost → CatBoost) Meilleur contre meilleur (RandomForest → CatBoost)
Variante    Mesure                                                                                                                                                                                            
Complet     Précision au rappel de 50%            ❌ perte  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                ➖ dans le bruit                                                NaN
            Précision au rappel de 60%    ➖ dans le bruit  ➖ dans le bruit      ✅ gain réel  ➖ dans le bruit                                ➖ dans le bruit                                                NaN
            Précision au rappel de 70%    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                ➖ dans le bruit                                                NaN
            Précision au rappel de 80%        ✅ gain réel      ✅ gain réel  ➖ dans le bruit      ✅ gain réel                                    ✅ gain réel                                                NaN
            Précision moyenne                     ❌ perte  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                ➖ dans le bruit                                                NaN
            Rappel, règle marginale               ❌ perte  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                ➖ dans le bruit                                                NaN
            Précision, règle marginale    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                ➖ dans le bruit                                                NaN
PAY_n seuls Précision au rappel de 50%    ➖ dans le bruit          ❌ perte      ✅ gain réel  ➖ dans le bruit                                            NaN                                    ➖ dans le bruit
            Précision au rappel de 60%    ➖ dans le bruit      ✅ gain réel  ➖ dans le bruit  ➖ dans le bruit                                            NaN                                    ➖ dans le bruit
            Précision au rappel de 70%    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit          ❌ perte                                            NaN                                    ➖ dans le bruit
            Précision au rappel de 80%            ❌ perte  ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit                                            NaN                                    ➖ dans le bruit
            Précision moyenne             ➖ dans le bruit  ➖ dans le bruit      ✅ gain réel  ➖ dans le bruit                                            NaN                                    ➖ dans le bruit
            Rappel, règle marginale       ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit          ❌ perte                                            NaN                                    ➖ dans le bruit
            Précision, règle marginale    ➖ dans le bruit  ➖ dans le bruit  ➖ dans le bruit      ✅ gain réel                                            NaN                                            ❌ perte

**Détail, pli par pli : jeu de base face au jeu avec correctifs**

Jeu de base (%)  Avec correctifs (%)    Gain (pts) Plis gagnés          Verdict
Variante    Modèle                                             Mesure                                                                                                     
Complet     LogisticRegression                                 Précision au rappel de 50%             52.5                 52.2  -0.29 ± 0.29     1 sur 5          ❌ perte
                                                               Précision au rappel de 60%             45.3                 45.3  -0.01 ± 0.25     2 sur 5  ➖ dans le bruit
                                                               Précision au rappel de 70%             37.5                 37.6  +0.03 ± 0.30     3 sur 5  ➖ dans le bruit
                                                               Précision au rappel de 80%             31.4                 31.6  +0.18 ± 0.16     4 sur 5      ✅ gain réel
                                                               Précision moyenne                      49.9                 49.8  -0.13 ± 0.09     0 sur 5          ❌ perte
                                                               Rappel, règle marginale                30.0                 28.7  -1.28 ± 1.18     2 sur 5          ❌ perte
                                                               Précision, règle marginale             62.4                 63.1  +0.61 ± 0.67     4 sur 5  ➖ dans le bruit
            MLPClassifier                                      Précision au rappel de 50%             57.7                 56.9  -0.76 ± 0.99     2 sur 5  ➖ dans le bruit
                                                               Précision au rappel de 60%             48.8                 49.3  +0.51 ± 0.58     4 sur 5  ➖ dans le bruit
                                                               Précision au rappel de 70%             40.7                 40.7  +0.02 ± 0.27     3 sur 5  ➖ dans le bruit
                                                               Précision au rappel de 80%             33.7                 34.2  +0.54 ± 0.50     5 sur 5      ✅ gain réel
                                                               Précision moyenne                      54.0                 54.0  +0.01 ± 0.14     2 sur 5  ➖ dans le bruit
                                                               Rappel, règle marginale                35.5                 34.9  -0.59 ± 3.28     4 sur 5  ➖ dans le bruit
                                                               Précision, règle marginale             66.4                 67.6  +1.20 ± 2.06     3 sur 5  ➖ dans le bruit
            RandomForest                                       Précision au rappel de 50%             57.4                 57.5  +0.02 ± 0.36     2 sur 5  ➖ dans le bruit
                                                               Précision au rappel de 60%             49.6                 50.2  +0.54 ± 0.12     5 sur 5      ✅ gain réel
                                                               Précision au rappel de 70%             41.5                 41.4  -0.14 ± 0.40     1 sur 5  ➖ dans le bruit
                                                               Précision au rappel de 80%             34.4                 34.3  -0.06 ± 0.34     2 sur 5  ➖ dans le bruit
                                                               Précision moyenne                      55.1                 55.1  -0.03 ± 0.18     2 sur 5  ➖ dans le bruit
                                                               Rappel, règle marginale                38.1                 36.8  -1.33 ± 2.90     3 sur 5  ➖ dans le bruit
                                                               Précision, règle marginale             63.9                 64.8  +0.91 ± 1.39     4 sur 5  ➖ dans le bruit
            CatBoost                                           Précision au rappel de 50%             58.2                 58.1  -0.12 ± 0.13     1 sur 5  ➖ d

**Comment lire** :
- **Précision à rappel égal** : à chaque ligne, les deux essais détectent la même part des défauts ; un gain positif veut dire que, pour détecter autant de défauts, le jeu corrigé signale moins de bons clients. C'est la réponse directe à la question de l'essai.
- **Précision moyenne** : le même constat résumé sur tous les rappels.
- **Règle marginale** : les deux essais ne détectent pas forcément la même part des défauts ; rappel et précision se lisent ensemble (un rappel plus haut à précision égale est un gain).
- **Verdict** : « dans le bruit » veut dire que les correctifs ne changent pas la précision au-delà de ce qui varie d'un pli à l'autre. Les 30 000 clients comprennent le contentieux, que les `PAY_n` repèrent facilement dans les deux essais.

## 7. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*